# 목표

2023년 1월 1일 기준
코스피 종목 중 시가총액 5000억 이상인 종목에 해당하여,

아래의 데이터에 의해서 각 종목의

1. 신용잔고, 신용비율
2. 외국인, 기관 순매수,순매도 데이터
3. 공매도 잔고, 대차잔고
4. 거래대금 , 거래량

에 대해서 (1) 각 종목의 주가가 어떤 흐름을 나타내는지.

또한 (2) KOSPI 200 지수에 대하여, 지수는 어떤 흐름을 나타내는지.

## 결과물

1. 2023-01-02 현재 코스피 종목 중 시가총액 5000억 이상 종목 (287개)
   - 시총5000억이상_20230102.csv
2. 외국인/기관 순매수 데이터
   - 기관_외국인_순매수_20230102_20251231.csv
3. 공매도 잔고
   - 공매도잔고_20230102_20251231.csv
4. 거래대금, 거래량
   - OHLCV_20230102_20251231.csv
5. 대차 잔고
   - 대차잔고_20230102_20251231.csv
6. 신용잔고, 대주잔고
   - 신용잔고_대주잔고_20230102_20251231.csv

## 1. 종목 코드 가져오기

2023년 1월초 기준 코스피 종목 중 시가총액 5000억 이상 종목


In [7]:
# pip install -U pykrx pandas

import pandas as pd
from pykrx import stock


def export_kospi_mcap_universe(date: str, min_mcap_won: int, out_csv: str) -> pd.DataFrame:
    """
    KRX > 통계 > 기본 통계 > 주식 > 종목 시세 > 전종목 시세 (KOSPI, 특정일)
    성격의 스냅샷에서 '시가총액' 기준으로 필터링해 CSV 저장.

    - date: 'YYYYMMDD'
    - min_mcap_won: 최소 시가총액(원). 예) 5000억 = 500_000_000_000
    """
    # 1) 전종목(코스피) 시가총액 스냅샷
    df = stock.get_market_cap_by_ticker(date, market="KOSPI")  # index = ticker

    if df is None or df.empty:
        raise RuntimeError(f"데이터가 비어있습니다. 휴장일/입력일자 오류 가능: {date}")

    if "시가총액" not in df.columns:
        raise KeyError(f"'시가총액' 컬럼이 없습니다. columns={list(df.columns)}")

    # 2) 시총 필터 (>= 5,000억)
    df_f = df[df["시가총액"] >= min_mcap_won].copy()

    # 3) 종목명 매핑 (티커 -> 이름)
    # (get_market_cap_by_ticker 결과에 종목명이 항상 포함되지 않아서, 매핑 방식이 가장 안전합니다.)
    df_f["종목명"] = [stock.get_market_ticker_name(t) for t in df_f.index]

    # 4) 결과 정리: 종목코드/종목명/시가총액
    out = (
        df_f[["종목명", "시가총액"]]
        .rename(columns={"시가총액": "시가총액(원)"})
        .reset_index()
        .rename(columns={"index": "종목코드"})
        .sort_values("시가총액(원)", ascending=False)
        .reset_index(drop=True)
    )

    # 종목코드 앞에 "A" 붙이기
    out["티커"] = "A" + out["티커"].astype(str)
    
    # 5) CSV 저장 (엑셀 호환: utf-8-sig)
    out.to_csv(out_csv, index=False, encoding="utf-8-sig")

    print(f"[OK] date={date} | min_mcap={min_mcap_won:,.0f}원 | n={len(out):,}")
    print(f"[SAVED] {out_csv}")
    return out


if __name__ == "__main__":
    date = "20230102"
    min_mcap = 500_000_000_000  # 5,000억 원
    out_csv = "시총5000억이상_20230102.csv"

    df_out = export_kospi_mcap_universe(date, min_mcap, out_csv)
    print(df_out.head(10))


[OK] date=20230102 | min_mcap=500,000,000,000원 | n=287
[SAVED] 시총5000억이상_20230102.csv
        티커       종목명          시가총액(원)
0  A005930      삼성전자  331322931525000
1  A373220  LG에너지솔루션  104364000000000
2  A207940  삼성바이오로직스   58860898000000
3  A000660    SK하이닉스   55109779030500
4  A051910      LG화학   42637775172000
5  A005935     삼성전자우   41802644360000
6  A006400     삼성SDI   41396247060000
7  A005380       현대차   33545905359000
8  A035420     NAVER   29446810757500
9  A000270        기아   24929845840500


## 2. 외국인/기관 순매수 데이터

In [ ]:
# pip install -U pykrx pandas

import os
import time
import pandas as pd
from pykrx import stock


def fetch_trading_value_by_date_chunked(
    ticker: str,
    start: str,   # "YYYYMMDD"
    end: str,     # "YYYYMMDD"
    on: str = "순매수",  # "순매수" | "매수" | "매도"
    chunk_months: int = 6,
    sleep_sec: float = 0.8,
    retries: int = 4,
) -> pd.DataFrame:
    """
    KRX(12009, 투자자별 거래실적-개별종목) 성격의 데이터:
    일별추이 / 거래대금 / (매도|매수|순매수)

    안정성을 위해 기간을 여러 청크로 쪼개 조회 후 concat.
    반환 DF index: 날짜, columns: 투자자(기관합계/개인/외국인합계/기타법인 등)
    """
    start_dt = pd.to_datetime(start)
    end_dt = pd.to_datetime(end)

    out = []
    cur = start_dt

    while cur <= end_dt:
        nxt = (cur + pd.DateOffset(months=chunk_months)) - pd.DateOffset(days=1)
        if nxt > end_dt:
            nxt = end_dt

        s = cur.strftime("%Y%m%d")
        e = nxt.strftime("%Y%m%d")

        last_err = None
        for attempt in range(1, retries + 1):
            try:
                df = stock.get_market_trading_value_by_date(s, e, ticker, on=on)
                out.append(df)
                break
            except Exception as ex:
                last_err = ex
                time.sleep(sleep_sec * (2 ** (attempt - 1)))
        else:
            raise RuntimeError(f"조회 실패: {ticker} {s}~{e} | 마지막 에러: {last_err}")

        time.sleep(sleep_sec)
        cur = nxt + pd.DateOffset(days=1)

    if not out:
        return pd.DataFrame()

    df_all = pd.concat(out).sort_index()
    df_all = df_all[~df_all.index.duplicated(keep="last")]
    return df_all


def load_universe_csv(path: str) -> pd.DataFrame:
    """
    유니버스 CSV를 읽고 ticker를 pykrx용 6자리로 정리.
    - 종목코드 컬럼이 'A005930'처럼 A가 붙어있으면 제거
    - 종목명 컬럼이 없으면 name은 빈 값으로 둠
    """
    df = pd.read_csv(path, dtype=str)

    # 컬럼명 후보 처리
    code_col = None
    for c in ["종목코드", "티커", "ticker", "code"]:
        if c in df.columns:
            code_col = c
            break
    if code_col is None:
        raise KeyError(f"CSV에 종목코드 컬럼이 없습니다. columns={list(df.columns)}")

    name_col = None
    for c in ["종목명", "name"]:
        if c in df.columns:
            name_col = c
            break

    df2 = pd.DataFrame()
    df2["ticker_raw"] = df[code_col].astype(str).str.strip()

    # 'A' 접두 제거 + 6자리 유지
    df2["ticker"] = (
        df2["ticker_raw"]
        .str.replace(" ", "", regex=False)
        .str.replace("\ufeff", "", regex=False)
    )
    df2["ticker"] = df2["ticker"].str[1:].where(df2["ticker"].str.startswith(("A", "a")), df2["ticker"])
    df2["ticker"] = df2["ticker"].str.zfill(6)

    if name_col:
        df2["name"] = df[name_col].astype(str).str.strip()
    else:
        df2["name"] = ""

    # 중복 제거
    df2 = df2.drop_duplicates(subset=["ticker"]).reset_index(drop=True)
    return df2


def build_trading_value_panel(
    universe_csv: str,
    start: str,
    end: str,
    out_csv: str,
    on: str = "순매수",
    cols_keep=("기관합계", "개인", "외국인합계"),
    chunk_months: int = 6,
    sleep_sec: float = 0.8,
    retries: int = 4,
    per_ticker_pause: float = 0.2,
    save_mode: str = "append"  # "append" or "memory"
) -> None:
    """
    유니버스의 모든 종목에 대해 투자자별 거래대금(순매수) 일별추이를 조회하여
    하나의 롱 포맷 CSV로 저장.

    save_mode:
      - "append": 종목별로 바로 out_csv에 append (대용량/중단 대비 유리)
      - "memory": 모두 메모리에 모았다가 한 번에 저장
    """
    uni = load_universe_csv(universe_csv)
    n = len(uni)

    # append 모드면 기존 파일 제거(원하면 주석 처리)
    if save_mode == "append" and os.path.exists(out_csv):
        os.remove(out_csv)

    all_frames = []

    for i, row in uni.iterrows():
        ticker = row["ticker"]
        name = row.get("name", "")

        print(f"[{i+1}/{n}] {ticker} {name} ...")

        try:
            df = fetch_trading_value_by_date_chunked(
                ticker=ticker,
                start=start,
                end=end,
                on=on,
                chunk_months=chunk_months,
                sleep_sec=sleep_sec,
                retries=retries
            )

            if df is None or df.empty:
                print(f"  -> EMPTY (skip)")
                time.sleep(per_ticker_pause)
                continue

            # 필요한 컬럼만 (없으면 생성)
            df2 = df.copy()
            for c in cols_keep:
                if c not in df2.columns:
                    df2[c] = pd.NA
            df2 = df2[list(cols_keep)]

            # 롱 포맷으로 정리
            out = df2.copy()
            out["date"] = out.index
            out = out.reset_index(drop=True)

            out["date"] = pd.to_datetime(out["date"]).dt.strftime("%Y-%m-%d")
            out.insert(0, "ticker", ticker)
            out.insert(1, "name", name)

            # ticker 앞에 'A' 붙이기
            out["ticker"] = "A" + out["ticker"].astype(str)

            # 컬럼 순서 강제
            col_order = ["date", "ticker", "name", "기관합계", "개인", "외국인합계"]
            out = out.reset_index(drop=True)
            out = out[col_order]
            
            # 저장
            if save_mode == "append":
                header = not os.path.exists(out_csv)
                out.to_csv(out_csv, index=False, encoding="utf-8-sig", mode="a", header=header)
            else:
                all_frames.append(out)

        except Exception as e:
            print(f"  -> FAILED: {ticker} | {e}")

        time.sleep(per_ticker_pause)

    if save_mode == "memory":
        if not all_frames:
            raise RuntimeError("수집 결과가 비어 있습니다.")
        final = pd.concat(all_frames, ignore_index=True)
        final.to_csv(out_csv, index=False, encoding="utf-8-sig")

    print(f"[DONE] saved -> {out_csv}")


if __name__ == "__main__":
    universe_csv = "시총5000억이상_20230102.csv"
    out_csv = "기관_외국인_순매수_20230102_20251231.csv"

    build_trading_value_panel(
        universe_csv=universe_csv,
        start="20230102",
        end="20251231",
        out_csv=out_csv,
        on="순매수",                 # "매수" / "매도"로도 가능
        cols_keep=("기관합계", "개인", "외국인합계"),
        chunk_months=6,
        sleep_sec=0.8,
        retries=4,
        per_ticker_pause=0.2,
        save_mode="append"           # 중단 대비 최강: 종목별 즉시 저장
    )


[1/287] 005930 삼성전자 ...
[2/287] 373220 LG에너지솔루션 ...
[3/287] 207940 삼성바이오로직스 ...
[4/287] 000660 SK하이닉스 ...
[5/287] 051910 LG화학 ...
[6/287] 005935 삼성전자우 ...
[7/287] 006400 삼성SDI ...
[8/287] 005380 현대차 ...
[9/287] 035420 NAVER ...
[10/287] 000270 기아 ...
[11/287] 035720 카카오 ...
[12/287] 005490 POSCO홀딩스 ...
[13/287] 068270 셀트리온 ...
[14/287] 028260 삼성물산 ...
[15/287] 105560 KB금융 ...
[16/287] 012330 현대모비스 ...
[17/287] 055550 신한지주 ...
[18/287] 003670 포스코퓨처엠 ...
[19/287] 096770 SK이노베이션 ...
[20/287] 066570 LG전자 ...
[21/287] 032830 삼성생명 ...
[22/287] 034730 SK ...
[23/287] 015760 한국전력 ...
[24/287] 033780 KT&G ...
[25/287] 086790 하나금융지주 ...
[26/287] 003550 LG ...
[27/287] 323410 카카오뱅크 ...
[28/287] 051900 LG생활건강 ...
[29/287] 010130 고려아연 ...
[30/287] 329180 HD현대중공업 ...
[31/287] 017670 SK텔레콤 ...
[32/287] 009150 삼성전기 ...
[33/287] 034020 두산에너빌리티 ...
[34/287] 036570 엔씨소프트 ...
[35/287] 011200 HMM ...
[36/287] 018260 삼성에스디에스 ...
[37/287] 010950 S-Oil ...
[38/287] 000810 삼성화재 ...
[39/287] 030200 KT ...
[40/2

# 3. 공매도 잔고

In [ ]:
# pip install -U pykrx pandas

import os
import time
import pandas as pd
from pykrx import stock

def normalize_ticker_for_csv(x: str) -> str:
    """
    어떤 형태든 최종적으로 'A005930' 형태로 통일
    """
    x = str(x).strip()

    # A 제거
    if x.startswith(("A", "a")):
        x = x[1:]

    # 숫자 6자리 복원
    x = x.zfill(6)

    return "A" + x

def load_universe_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str)

    code_col = None
    for c in ["종목코드", "티커", "ticker", "code"]:
        if c in df.columns:
            code_col = c
            break
    if code_col is None:
        raise KeyError(f"CSV에 종목코드 컬럼이 없습니다. columns={list(df.columns)}")

    name_col = None
    for c in ["종목명", "name"]:
        if c in df.columns:
            name_col = c
            break

    out = pd.DataFrame()
    out["ticker_raw"] = df[code_col].astype(str).str.strip()

    # 'A' 접두 제거 + 6자리
    out["ticker"] = out["ticker_raw"].str.replace("\ufeff", "", regex=False)
    out["ticker"] = out["ticker"].str[1:].where(out["ticker"].str.startswith(("A","a")), out["ticker"])
    out["ticker"] = out["ticker"].str.zfill(6)

    out["name"] = df[name_col].astype(str).str.strip() if name_col else ""
    out = out.drop_duplicates(subset=["ticker"]).reset_index(drop=True)
    return out


def fetch_shorting_balance_by_date_chunked(
    ticker: str,
    start: str,  # "YYYYMMDD"
    end: str,    # "YYYYMMDD"
    chunk_months: int = 6,
    sleep_sec: float = 0.8,
    retries: int = 4,
) -> pd.DataFrame:
    """
    종목별 공매도 잔고 현황 (pykrx)
    - stock.get_shorting_balance_by_date(start, end, ticker)
      columns 예: 공매도잔고, 상장주식수, 공매도금액, 시가총액, 비중
    """
    start_dt = pd.to_datetime(start)
    end_dt = pd.to_datetime(end)

    frames = []
    cur = start_dt
    while cur <= end_dt:
        nxt = (cur + pd.DateOffset(months=chunk_months)) - pd.DateOffset(days=1)
        if nxt > end_dt:
            nxt = end_dt

        s = cur.strftime("%Y%m%d")
        e = nxt.strftime("%Y%m%d")

        last_err = None
        for attempt in range(1, retries + 1):
            try:
                df = stock.get_shorting_balance_by_date(s, e, ticker)
                frames.append(df)
                break
            except Exception as ex:
                last_err = ex
                time.sleep(sleep_sec * (2 ** (attempt - 1)))
        else:
            raise RuntimeError(f"공매도 잔고 조회 실패: {ticker} {s}~{e} | {last_err}")

        time.sleep(sleep_sec)
        cur = nxt + pd.DateOffset(days=1)

    if not frames:
        return pd.DataFrame()

    df_all = pd.concat(frames).sort_index()
    df_all = df_all[~df_all.index.duplicated(keep="last")]
    return df_all


def build_shorting_balance_panel(
    universe_csv: str,
    start: str,
    end: str,
    out_csv: str = "공매도잔고_유니버스_20230102_20251231.csv",
    chunk_months: int = 6,
    sleep_sec: float = 0.8,
    retries: int = 4,
    per_ticker_pause: float = 0.2,
):
    uni = load_universe_csv(universe_csv)
    n = len(uni)

    if os.path.exists(out_csv):
        os.remove(out_csv)

    for i, row in uni.iterrows():
        ticker = row["ticker"]
        name = row.get("name", "")
        print(f"[{i+1}/{n}] {ticker} {name} ...")

        try:
            df = fetch_shorting_balance_by_date_chunked(
                ticker=ticker, start=start, end=end,
                chunk_months=chunk_months, sleep_sec=sleep_sec, retries=retries
            )
            if df.empty:
                print("  -> EMPTY (skip)")
                continue

            out = df.copy()
            out["date"] = out.index
            out = out.reset_index(drop=True)
            out["date"] = pd.to_datetime(out["date"]).dt.strftime("%Y-%m-%d")
            out.insert(1, "ticker", normalize_ticker_for_csv(ticker))
            out.insert(2, "name", name)

            col_order = ["date", "ticker", "name", "공매도잔고", "상장주식수", "공매도금액", "시가총액", "비중"]
            out = out[col_order]

            header = not os.path.exists(out_csv)
            out.to_csv(out_csv, index=False, encoding="utf-8-sig", mode="a", header=header)

        except Exception as e:
            print(f"  -> FAILED: {ticker} | {e}")

        time.sleep(per_ticker_pause)

    print(f"[DONE] saved -> {out_csv}")


if __name__ == "__main__":
    build_shorting_balance_panel(
        universe_csv="시총5000억이상_20230102.csv",
        start="20230102",
        end="20251231",
        out_csv="공매도잔고_20230102_20251231.csv",
    )


[1/287] 005930 삼성전자 ...
[2/287] 373220 LG에너지솔루션 ...
[3/287] 207940 삼성바이오로직스 ...
[4/287] 000660 SK하이닉스 ...
[5/287] 051910 LG화학 ...
[6/287] 005935 삼성전자우 ...
[7/287] 006400 삼성SDI ...
[8/287] 005380 현대차 ...
[9/287] 035420 NAVER ...
[10/287] 000270 기아 ...
[11/287] 035720 카카오 ...
[12/287] 005490 POSCO홀딩스 ...
[13/287] 068270 셀트리온 ...
[14/287] 028260 삼성물산 ...
[15/287] 105560 KB금융 ...
[16/287] 012330 현대모비스 ...
[17/287] 055550 신한지주 ...
[18/287] 003670 포스코퓨처엠 ...
[19/287] 096770 SK이노베이션 ...
[20/287] 066570 LG전자 ...
[21/287] 032830 삼성생명 ...
[22/287] 034730 SK ...
[23/287] 015760 한국전력 ...
[24/287] 033780 KT&G ...
[25/287] 086790 하나금융지주 ...
[26/287] 003550 LG ...
[27/287] 323410 카카오뱅크 ...
[28/287] 051900 LG생활건강 ...
[29/287] 010130 고려아연 ...
[30/287] 329180 HD현대중공업 ...
[31/287] 017670 SK텔레콤 ...
[32/287] 009150 삼성전기 ...
[33/287] 034020 두산에너빌리티 ...
[34/287] 036570 엔씨소프트 ...
[35/287] 011200 HMM ...
[36/287] 018260 삼성에스디에스 ...
[37/287] 010950 S-Oil ...
[38/287] 000810 삼성화재 ...
[39/287] 030200 KT ...
[40/2

## 4. 거래대금, 거래량

In [2]:
# pip install -U pykrx pandas

import os
import time
import pandas as pd
from pykrx import stock


def normalize_ticker_for_api(x: str) -> str:
    """pykrx 호출용: 6자리 숫자"""
    s = str(x).strip().replace("\ufeff", "")
    if s.lower().startswith("a"):
        s = s[1:]
    s = "".join(ch for ch in s if ch.isdigit())
    return s.zfill(6)


def normalize_ticker_for_csv(x: str) -> str:
    """CSV 저장용: A + 6자리"""
    return "A" + normalize_ticker_for_api(x)


def load_universe_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str)

    code_col = None
    for c in ["종목코드", "ticker", "code", "티커"]:
        if c in df.columns:
            code_col = c
            break
    if code_col is None:
        raise KeyError(f"유니버스 CSV에서 종목코드 컬럼을 찾지 못했습니다. columns={list(df.columns)}")

    name_col = None
    for c in ["종목명", "name", "종목"]:
        if c in df.columns:
            name_col = c
            break

    out = pd.DataFrame()
    out["ticker_api"] = df[code_col].apply(normalize_ticker_for_api)
    out["ticker"] = df[code_col].apply(normalize_ticker_for_csv)
    out["name"] = df[name_col].astype(str).str.strip() if name_col else ""
    out = out.drop_duplicates(subset=["ticker_api"]).reset_index(drop=True)
    return out


def fetch_ohlcv_chunked(
    ticker_api: str,
    start: str,   # YYYYMMDD
    end: str,     # YYYYMMDD
    chunk_months: int = 6,
    sleep_sec: float = 0.8,
    retries: int = 4,
) -> pd.DataFrame:
    """종목별 OHLCV(일자) 기간 청크 조회 후 concat"""
    start_dt = pd.to_datetime(start)
    end_dt = pd.to_datetime(end)

    frames = []
    cur = start_dt
    while cur <= end_dt:
        nxt = (cur + pd.DateOffset(months=chunk_months)) - pd.DateOffset(days=1)
        if nxt > end_dt:
            nxt = end_dt

        s = cur.strftime("%Y%m%d")
        e = nxt.strftime("%Y%m%d")

        last_err = None
        for attempt in range(1, retries + 1):
            try:
                df = stock.get_market_ohlcv_by_date(s, e, ticker_api)
                frames.append(df)
                break
            except Exception as ex:
                last_err = ex
                time.sleep(sleep_sec * (2 ** (attempt - 1)))
        else:
            raise RuntimeError(f"OHLCV 조회 실패: {ticker_api} {s}~{e} | 마지막 에러: {last_err}")

        time.sleep(sleep_sec)
        cur = nxt + pd.DateOffset(days=1)

    if not frames:
        return pd.DataFrame()

    df_all = pd.concat(frames).sort_index()
    df_all = df_all[~df_all.index.duplicated(keep="last")]
    return df_all


def build_ohlcv_value_panel(
    universe_csv: str,
    start: str,
    end: str,
    out_csv: str,
    chunk_months: int = 6,
    sleep_sec: float = 0.8,
    retries: int = 4,
    per_ticker_pause: float = 0.2,
):
    uni = load_universe_csv(universe_csv)
    n = len(uni)

    if os.path.exists(out_csv):
        os.remove(out_csv)

    for i, row in uni.iterrows():
        ticker_api = row["ticker_api"]     # 005930
        ticker_csv = row["ticker"]         # A005930
        name = row["name"]

        print(f"[{i+1}/{n}] {ticker_api} {name} ...")

        try:
            df = fetch_ohlcv_chunked(
                ticker_api=ticker_api,
                start=start,
                end=end,
                chunk_months=chunk_months,
                sleep_sec=sleep_sec,
                retries=retries
            )

            if df.empty:
                print("  -> EMPTY (skip)")
                time.sleep(per_ticker_pause)
                continue

            # 필수 컬럼 체크 (거래대금은 계산)
            required = ["시가", "고가", "저가", "종가", "거래량"]
            missing = [c for c in required if c not in df.columns]
            if missing:
                raise KeyError(f"필수 컬럼 누락: {missing}, columns={list(df.columns)}")

            out = df[["시가", "고가", "저가", "종가", "거래량"]].copy()

            # ✅ 거래대금 계산 (정석)
            out["거래대금"] = out["종가"] * out["거래량"]

            # date 컬럼 만들기
            out["date"] = out.index
            out = out.reset_index(drop=True)
            out["date"] = pd.to_datetime(out["date"]).dt.strftime("%Y-%m-%d")

            # 키 컬럼을 앞으로
            out.insert(1, "ticker", ticker_csv)
            out.insert(2, "name", name)

            # 컬럼 순서 강제
            out = out[["date", "ticker", "name", "시가", "고가", "저가", "종가", "거래량", "거래대금"]]

            header = not os.path.exists(out_csv)
            out.to_csv(out_csv, index=False, encoding="utf-8-sig", mode="a", header=header)

            print(f"  -> OK rows={len(out):,}")

        except Exception as e:
            print(f"  -> FAILED: {ticker_api} | {e}")

        time.sleep(per_ticker_pause)

    print(f"[DONE] saved -> {out_csv}")


if __name__ == "__main__":
    build_ohlcv_value_panel(
        universe_csv="시총5000억이상_20230102.csv",
        start="20230102",
        end="20251231",
        out_csv="OHLCV_20230102_20251231.csv",
        chunk_months=6,
        sleep_sec=0.8,
        retries=4,
        per_ticker_pause=0.2,
    )


[1/287] 005930 삼성전자 ...
  -> OK rows=731
[2/287] 373220 LG에너지솔루션 ...
  -> OK rows=731
[3/287] 207940 삼성바이오로직스 ...
  -> OK rows=731
[4/287] 000660 SK하이닉스 ...
  -> OK rows=731
[5/287] 051910 LG화학 ...
  -> OK rows=731
[6/287] 005935 삼성전자우 ...
  -> OK rows=731
[7/287] 006400 삼성SDI ...
  -> OK rows=731
[8/287] 005380 현대차 ...
  -> OK rows=731
[9/287] 035420 NAVER ...
  -> OK rows=731
[10/287] 000270 기아 ...
  -> OK rows=731
[11/287] 035720 카카오 ...
  -> OK rows=731
[12/287] 005490 POSCO홀딩스 ...
  -> OK rows=731
[13/287] 068270 셀트리온 ...
  -> OK rows=731
[14/287] 028260 삼성물산 ...
  -> OK rows=731
[15/287] 105560 KB금융 ...
  -> OK rows=731
[16/287] 012330 현대모비스 ...
  -> OK rows=731
[17/287] 055550 신한지주 ...
  -> OK rows=731
[18/287] 003670 포스코퓨처엠 ...
  -> OK rows=731
[19/287] 096770 SK이노베이션 ...
  -> OK rows=731
[20/287] 066570 LG전자 ...
  -> OK rows=731
[21/287] 032830 삼성생명 ...
  -> OK rows=731
[22/287] 034730 SK ...
  -> OK rows=731
[23/287] 015760 한국전력 ...
  -> OK rows=731
[24/287] 033780 KT&G ...
 

## 5. 대차잔고

In [7]:
# pip install -U pandas requests

import os
import time
import requests
import pandas as pd
from typing import List, Dict, Any, Optional


SERVICE_KEY = "c6263b287fd24820c4c11cfe89517ebfeaa2a6b05948f5adf8215d5a068b0e9b"  # 본인 키로 교체

LNB_DETAIL_URL = "https://apis.data.go.kr/1160100/service/GetCMStckLnbInfoService/getStckLnbDetail"
DEFAULT_TIMEOUT = 30


def normalize_ticker_for_api(x: str) -> str:
    """API 호출용: 6자리 숫자"""
    s = str(x).strip().replace("\ufeff", "")
    if s.lower().startswith("a"):
        s = s[1:]
    s = "".join(ch for ch in s if ch.isdigit())
    return s.zfill(6)


def normalize_ticker_for_csv(x: str) -> str:
    """CSV 저장용: A + 6자리"""
    return "A" + normalize_ticker_for_api(x)


def load_universe_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str)

    code_col = None
    for c in ["종목코드", "ticker", "code", "티커"]:
        if c in df.columns:
            code_col = c
            break
    if code_col is None:
        raise KeyError(f"유니버스 CSV에서 종목코드 컬럼을 찾지 못했습니다. columns={list(df.columns)}")

    name_col = None
    for c in ["종목명", "name", "종목"]:
        if c in df.columns:
            name_col = c
            break

    out = pd.DataFrame()
    out["ticker_api"] = df[code_col].apply(normalize_ticker_for_api)
    out["ticker"] = df[code_col].apply(normalize_ticker_for_csv)
    out["name"] = df[name_col].astype(str).str.strip() if name_col else ""
    out = out.drop_duplicates(subset=["ticker_api"]).reset_index(drop=True)
    return out


def _items(data: Dict[str, Any]) -> List[Dict[str, Any]]:
    item = ((data.get("response") or {}).get("body") or {}).get("items", {}).get("item")
    if item is None:
        return []
    if isinstance(item, list):
        return item
    if isinstance(item, dict):
        return [item]
    return []


def fetch_lnb_detail_all_pages(
    ticker_api: str,
    begin: str,
    end: str,
    num_rows: int = 5000,
    sleep_sec: float = 0.2,
    retries: int = 4,
) -> pd.DataFrame:
    """
    종목 1개에 대해 기간(begin~end) 대차상세를 페이지네이션으로 전부 수집
    """
    params = {
        "serviceKey": SERVICE_KEY,
        "resultType": "json",
        "pageNo": 1,
        "numOfRows": num_rows,
        "stckItmsCd": ticker_api,
        "beginBasDt": begin,
        "endBasDt": end,
    }
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json",
    }

    all_rows: List[Dict[str, Any]] = []
    page = 1

    while True:
        params["pageNo"] = page

        last_err: Optional[Exception] = None
        for attempt in range(1, retries + 1):
            try:
                r = requests.get(LNB_DETAIL_URL, params=params, headers=headers, timeout=DEFAULT_TIMEOUT)
                if r.status_code != 200:
                    # 에러 메시지 확인용(원인 추적에 중요)
                    raise RuntimeError(f"HTTP {r.status_code} | {r.text[:300]}")
                data = r.json()
                rows = _items(data)
                all_rows.extend(rows)

                body = (data.get("response") or {}).get("body") or {}
                total = body.get("totalCount")

                # 종료 조건
                if total is not None:
                    if len(all_rows) >= int(total):
                        return pd.DataFrame(all_rows)
                else:
                    # totalCount가 없으면 rows가 비면 종료
                    if not rows:
                        return pd.DataFrame(all_rows)

                # 다음 페이지
                page += 1
                time.sleep(sleep_sec)
                break

            except Exception as ex:
                last_err = ex
                time.sleep(sleep_sec * (2 ** (attempt - 1)))
        else:
            raise RuntimeError(f"대차 조회 실패: {ticker_api} p{page} | {last_err}")


def transform_lnb_df(df: pd.DataFrame, ticker_csv: str, name: str) -> pd.DataFrame:
    """
    - date 오름차순 정렬
    - 중복 컬럼 제거
    - 컬럼명 한글화
    - date,ticker,name 앞으로
    """
    if df.empty:
        return df

    # basDt -> date
    if "basDt" in df.columns:
        df["date"] = pd.to_datetime(df["basDt"].astype(str), format="%Y%m%d", errors="coerce").dt.strftime("%Y-%m-%d")
    elif "date" not in df.columns:
        raise KeyError(f"날짜 컬럼이 없습니다. columns={list(df.columns)}")

    # 숫자형 변환(가능하면)
    for col in ["cclStckCnt", "rdptStckCnt", "balnStckCnt", "balnStckAmt"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # 컬럼명 한글화
    rename_map = {
        "cclStckCnt": "대차체결주식수",
        "rdptStckCnt": "상환주식수",
        "balnStckCnt": "대차잔고주식수",
        "balnStckAmt": "대차잔고금액",
    }
    df = df.rename(columns=rename_map)

    # 중복/불필요 컬럼 제거
    drop_cols = [c for c in ["basDt", "mrktClsfNm", "stckItmsNm", "stckItmsCd"] if c in df.columns]
    df = df.drop(columns=drop_cols, errors="ignore")

    # 키 컬럼 추가(또는 덮어쓰기)
    df["ticker"] = ticker_csv
    df["name"] = name

    # 필요한 컬럼만 남기고 순서 고정
    keep = ["date", "ticker", "name", "대차체결주식수", "상환주식수", "대차잔고주식수", "대차잔고금액"]
    keep = [c for c in keep if c in df.columns]
    df = df[keep].copy()

    # 날짜 오름차순
    df = df.sort_values("date", ascending=True).reset_index(drop=True)

    return df


def build_lnb_panel_universe(
    universe_csv: str,
    begin: str,
    end: str,
    out_csv: str = "대차잔고_시총5000억이상_20230102_20251231.csv",
    num_rows: int = 5000,
    sleep_sec: float = 0.2,
    retries: int = 4,
    per_ticker_pause: float = 0.15,
):
    uni = load_universe_csv(universe_csv)
    n = len(uni)

    # append 저장을 위해 기존 파일 삭제(원하면 주석 처리)
    if os.path.exists(out_csv):
        os.remove(out_csv)

    header = True

    for i, row in uni.iterrows():
        ticker_api = row["ticker_api"]  # 005930
        ticker_csv = row["ticker"]      # A005930
        name = row["name"]

        print(f"[{i+1}/{n}] {ticker_api} {name} ...")

        try:
            raw = fetch_lnb_detail_all_pages(
                ticker_api=ticker_api,
                begin=begin,
                end=end,
                num_rows=num_rows,
                sleep_sec=sleep_sec,
                retries=retries,
            )

            if raw.empty:
                print("  -> EMPTY")
                time.sleep(per_ticker_pause)
                continue

            out = transform_lnb_df(raw, ticker_csv=ticker_csv, name=name)

            # 저장
            out.to_csv(out_csv, index=False, encoding="utf-8-sig", mode="a", header=header)
            header = False

            print(f"  -> OK rows={len(out):,}")

        except Exception as e:
            print(f"  -> FAILED: {ticker_api} | {e}")

        time.sleep(per_ticker_pause)

    print(f"[DONE] saved -> {out_csv}")


if __name__ == "__main__":
    build_lnb_panel_universe(
        universe_csv="시총5000억이상_20230102.csv",
        begin="20230102",
        end="20251231",
        out_csv="대차잔고_20230102_20251231.csv",
        num_rows=5000,
        sleep_sec=0.2,
        retries=4,
        per_ticker_pause=0.15,
    )


[1/287] 005930 삼성전자 ...
  -> OK rows=731
[2/287] 373220 LG에너지솔루션 ...
  -> OK rows=731
[3/287] 207940 삼성바이오로직스 ...
  -> OK rows=731
[4/287] 000660 SK하이닉스 ...
  -> OK rows=731
[5/287] 051910 LG화학 ...
  -> OK rows=731
[6/287] 005935 삼성전자우 ...
  -> OK rows=731
[7/287] 006400 삼성SDI ...
  -> OK rows=731
[8/287] 005380 현대차 ...
  -> OK rows=731
[9/287] 035420 NAVER ...
  -> OK rows=731
[10/287] 000270 기아 ...
  -> OK rows=731
[11/287] 035720 카카오 ...
  -> OK rows=731
[12/287] 005490 POSCO홀딩스 ...
  -> OK rows=731
[13/287] 068270 셀트리온 ...
  -> OK rows=731
[14/287] 028260 삼성물산 ...
  -> OK rows=731
[15/287] 105560 KB금융 ...
  -> OK rows=731
[16/287] 012330 현대모비스 ...
  -> OK rows=731
[17/287] 055550 신한지주 ...
  -> OK rows=731
[18/287] 003670 포스코퓨처엠 ...
  -> OK rows=731
[19/287] 096770 SK이노베이션 ...
  -> OK rows=731
[20/287] 066570 LG전자 ...
  -> OK rows=731
[21/287] 032830 삼성생명 ...
  -> OK rows=731
[22/287] 034730 SK ...
  -> OK rows=731
[23/287] 015760 한국전력 ...
  -> OK rows=731
[24/287] 033780 KT&G ...
 

## 6. 신용잔고, 대주잔고

KRX에서는 데이터가 없어서  
한투 KIS API를 사용하여 추출함

In [32]:
import yaml
import requests
import json

with open('../config.yaml', 'r') as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

api_key = config['hantu']['api_key']
secret_key = config['hantu']['secret_key']

base_url = 'https://openapivts.koreainvestment.com:29443'

headers = {"content-type":"application/json"}
body = {
        "grant_type":"client_credentials",
        "appkey":api_key,
        "appsecret":secret_key,
        }
url = base_url + '/oauth2/tokenP'
res = requests.post(url, headers=headers, data=json.dumps(body)).json()

access_token = res['access_token']


In [33]:
import os
import time
import requests
import pandas as pd
from typing import Set, Optional

# =========================================================
# 설정값
# =========================================================
APP_KEY = api_key
APP_SECRET = secret_key

# 모의(VTS) / 실전 선택
IS_VTS = True
BASE_URL = "https://openapivts.koreainvestment.com:29443" if IS_VTS else "https://openapi.koreainvestment.com:9443"
TR_ID = "FHPST04760000"  # 국내주식-110 (VTS에서 동작 확인)

UNIVERSE_CSV = "시총5000억이상_20230102.csv"
OUT_CSV = "신용잔고_대주잔고_20230102_20251231.csv"

START_TRADE = "20230102"   # 분석 시작 거래일(YYYYMMDD)
END_TRADE   = "20251231"   # 분석 종료 거래일(YYYYMMDD)

# 결제일 기준 시작점: END_TRADE + 14일(연말/연휴 누락 방지)
END_SETTLEMENT = (pd.to_datetime(END_TRADE, format="%Y%m%d") + pd.Timedelta(days=14)).strftime("%Y%m%d")

# 속도/재시도
PAUSE_BETWEEN_CALLS = 0.25      # 일반 호출 간격(초)
MAX_CALLS_PER_TICKER = 80       # 종목당 최대 chunk 호출(3년이면 보통 25~35면 충분)
CHUNK_RETRIES = 8               # chunk 요청 재시도
RATE_BASE_SLEEP = 0.8           # 레이트리밋 기본 대기(초)


# =========================================================
# 유틸: 종목코드 정규화
# =========================================================
def normalize_ticker_for_api(x: str) -> str:
    s = str(x).strip().replace("\ufeff", "")
    if s.lower().startswith("a"):
        s = s[1:]
    s = "".join(ch for ch in s if ch.isdigit())
    return s.zfill(6)

def normalize_ticker_for_csv(x: str) -> str:
    return "A" + normalize_ticker_for_api(x)

def load_universe_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str)

    code_col = None
    for c in ["종목코드", "ticker", "code", "티커"]:
        if c in df.columns:
            code_col = c
            break
    if code_col is None:
        raise ValueError(f"유니버스 CSV에서 종목코드 컬럼을 찾지 못했습니다. columns={list(df.columns)}")

    name_col = None
    for c in ["종목명", "name", "종목"]:
        if c in df.columns:
            name_col = c
            break

    out = pd.DataFrame()
    out["ticker_api"] = df[code_col].apply(normalize_ticker_for_api)
    out["ticker"] = df[code_col].apply(normalize_ticker_for_csv)
    out["name"] = df[name_col].astype(str).str.strip() if name_col else ""
    return out.drop_duplicates("ticker_api").reset_index(drop=True)

def load_done_tickers(out_csv: str) -> Set[str]:
    if not os.path.exists(out_csv):
        return set()
    try:
        done = pd.read_csv(out_csv, usecols=["ticker"], dtype=str)["ticker"].dropna().unique().tolist()
        return set(done)
    except Exception:
        return set()


# =========================================================
# 토큰 발급(강화판)
# =========================================================
def get_access_token_robust(base_url: str, app_key: str, app_secret: str, retries: int = 5) -> str:
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0"})

    endpoints = ["/oauth2/tokenP", "/oauth2/token"]
    payload = {
        "grant_type": "client_credentials",
        "appkey": app_key,
        "appsecret": app_secret,
    }

    last_err = None
    for ep in endpoints:
        url = f"{base_url}{ep}"
        for i in range(1, retries + 1):
            try:
                r = session.post(
                    url,
                    data=payload,
                    headers={"content-type": "application/x-www-form-urlencoded"},
                    timeout=30,
                )
                if r.status_code == 200:
                    j = r.json()
                    if "access_token" in j:
                        return j["access_token"]
                    raise RuntimeError(f"토큰 응답에 access_token이 없음: {j}")

                last_err = RuntimeError(f"token failed {r.status_code} @ {ep}: {r.text[:300]}")
                time.sleep(min(2 ** (i - 1), 10))
            except Exception as e:
                last_err = e
                time.sleep(min(2 ** (i - 1), 10))

    raise RuntimeError(f"토큰 발급 최종 실패: {last_err}")


# =========================================================
# KIS 신용잔고 chunk 호출(레이트리밋 대응)
# =========================================================
def fetch_credit_chunk_rl(
    base_url: str,
    token: str,
    app_key: str,
    app_secret: str,
    ticker_api: str,
    settlement_date: str,    # fid_input_date_1
    tr_id: str,
    max_retries: int = CHUNK_RETRIES,
    base_sleep: float = RATE_BASE_SLEEP,
) -> pd.DataFrame:
    url = f"{base_url}/uapi/domestic-stock/v1/quotations/daily-credit-balance"
    headers = {
        "content-type": "application/json; charset=utf-8",
        "authorization": f"Bearer {token}",
        "appkey": app_key,
        "appsecret": app_secret,
        "tr_id": tr_id,
    }
    params = {
        "fid_cond_mrkt_div_code": "J",
        "fid_cond_scr_div_code": "20476",
        "fid_input_iscd": ticker_api,
        "fid_input_date_1": settlement_date,
    }

    last_err: Optional[Exception] = None
    for attempt in range(1, max_retries + 1):
        r = requests.get(url, headers=headers, params=params, timeout=30)

        # 200이면 JSON 검사
        if r.status_code == 200:
            data = r.json()
            if data.get("rt_cd") == "0":
                out = data.get("output", [])
                if isinstance(out, dict):
                    out = [out]
                return pd.DataFrame(out)

            msg1 = str(data.get("msg1", ""))
            # 레이트리밋 문구면 백오프
            if "초당" in msg1 or "거래건수" in msg1 or data.get("msg_cd") in ("EGW00201",):
                sleep = min(base_sleep * (2 ** (attempt - 1)), 20)
                time.sleep(sleep)
                continue

            raise RuntimeError(f"API rt_cd={data.get('rt_cd')} msg_cd={data.get('msg_cd')} msg1={msg1}")

        # 429/500에 레이트리밋이 섞여 오는 케이스
        body = r.text[:500]
        if r.status_code in (429, 500) and ("초당" in body or "거래건수" in body or "EGW00201" in body):
            sleep = min(base_sleep * (2 ** (attempt - 1)), 20)
            time.sleep(sleep)
            last_err = RuntimeError(f"rate limited {r.status_code}: {body}")
            continue

        last_err = RuntimeError(f"HTTP {r.status_code}: {body}")
        break

    raise RuntimeError(f"chunk 요청 실패(재시도 초과): {last_err}")


# =========================================================
# 종목 1개: 30개 제한을 반복 호출로 3년치 누적
# =========================================================
def collect_credit_full_range_for_ticker(
    base_url: str,
    token: str,
    app_key: str,
    app_secret: str,
    tr_id: str,
    ticker_api: str,
    start_trade: str,
    end_trade: str,
    end_settlement: str,
    pause: float = PAUSE_BETWEEN_CALLS,
    max_calls: int = MAX_CALLS_PER_TICKER,
) -> pd.DataFrame:
    start_dt = pd.to_datetime(start_trade, format="%Y%m%d")
    end_dt = pd.to_datetime(end_trade, format="%Y%m%d")

    next_settlement = end_settlement
    chunks = []
    seen_next = set()

    for k in range(max_calls):
        if next_settlement in seen_next:
            break
        seen_next.add(next_settlement)

        df = fetch_credit_chunk_rl(
            base_url=base_url,
            token=token,
            app_key=app_key,
            app_secret=app_secret,
            ticker_api=ticker_api,
            settlement_date=next_settlement,
            tr_id=tr_id,
        )

        if df.empty:
            break

        df["deal_dt"] = pd.to_datetime(df["deal_date"].astype(str), format="%Y%m%d", errors="coerce")
        df["stlm_dt"] = pd.to_datetime(df["stlm_date"].astype(str), format="%Y%m%d", errors="coerce")
        df = df.dropna(subset=["deal_dt", "stlm_dt"])

        # 거래일 기준 범위 필터
        in_range = df[(df["deal_dt"] >= start_dt) & (df["deal_dt"] <= end_dt)].copy()
        if not in_range.empty:
            chunks.append(in_range)

        # 종료 조건: start 이하로 내려감
        if df["deal_dt"].min() <= start_dt:
            break

        # 다음 결제일자 = 이번 응답 가장 오래된 결제일자 - 1일
        min_stlm = df["stlm_dt"].min()
        next_settlement = (min_stlm - pd.Timedelta(days=1)).strftime("%Y%m%d")

        time.sleep(pause)

    if not chunks:
        return pd.DataFrame()

    out = pd.concat(chunks, ignore_index=True)
    out = out.sort_values("deal_dt").drop_duplicates(subset=["deal_date"], keep="last").reset_index(drop=True)
    return out


# =========================================================
# 변환: date,ticker,name,... 한글 컬럼 + 정렬
# =========================================================
def transform_credit_df(df: pd.DataFrame, ticker_csv: str, name: str) -> pd.DataFrame:
    if df.empty:
        return df

    rename_map = {
        "whol_loan_rmnd_stcn": "신용잔고주",
        "whol_loan_rmnd_amt": "신용잔고금액",
        "whol_loan_rmnd_rate": "신용비율",
        "whol_stln_rmnd_stcn": "대주잔고주",
        "whol_stln_rmnd_amt": "대주잔고금액",
        "whol_stln_rmnd_rate": "대주비율",
    }
    for k, v in rename_map.items():
        if k in df.columns:
            df = df.rename(columns={k: v})

    df["date"] = df["deal_dt"].dt.strftime("%Y-%m-%d")
    df.insert(0, "date", df.pop("date"))
    df.insert(1, "ticker", ticker_csv)
    df.insert(2, "name", name)

    keep = ["date", "ticker", "name",
            "신용잔고주", "신용잔고금액", "신용비율",
            "대주잔고주", "대주잔고금액", "대주비율"]
    keep = [c for c in keep if c in df.columns]
    df = df[keep].sort_values("date", ascending=True).reset_index(drop=True)
    return df


# =========================================================
# 전체 빌더(287종목)
# =========================================================
def build_credit_panel():
    uni = load_universe_csv(UNIVERSE_CSV)
    done = load_done_tickers(OUT_CSV)
    header = not os.path.exists(OUT_CSV)

    token = get_access_token_robust(BASE_URL, APP_KEY, APP_SECRET)

    print(f"[INFO] universe={len(uni)} done={len(done)}")
    print(f"[INFO] start_trade={START_TRADE} end_trade={END_TRADE} end_settlement={END_SETTLEMENT}")
    print(f"[INFO] out_csv={OUT_CSV}")

    for i, row in uni.iterrows():
        ticker_api = row["ticker_api"]
        ticker_csv = row["ticker"]
        name = row["name"]

        if ticker_csv in done:
            continue

        print(f"[{i+1}/{len(uni)}] {ticker_api} {name} ...")

        try:
            raw = collect_credit_full_range_for_ticker(
                base_url=BASE_URL,
                token=token,
                app_key=APP_KEY,
                app_secret=APP_SECRET,
                tr_id=TR_ID,
                ticker_api=ticker_api,
                start_trade=START_TRADE,
                end_trade=END_TRADE,
                end_settlement=END_SETTLEMENT,
                pause=PAUSE_BETWEEN_CALLS,
                max_calls=MAX_CALLS_PER_TICKER,
            )
            out = transform_credit_df(raw, ticker_csv, name)

            if out.empty:
                print("  -> EMPTY")
            else:
                out.to_csv(OUT_CSV, index=False, encoding="utf-8-sig", mode="a", header=header)
                header = False
                print(f"  -> OK rows={len(out):,} date_min={out['date'].min()} date_max={out['date'].max()}")

            done.add(ticker_csv)

        except Exception as e:
            print(f"  -> FAILED: {ticker_api} | {e}")

        time.sleep(PAUSE_BETWEEN_CALLS)

    print(f"[DONE] saved -> {OUT_CSV}")


if __name__ == "__main__":
    build_credit_panel()


[INFO] universe=287 done=0
[INFO] start_trade=20230102 end_trade=20251231 end_settlement=20260114
[INFO] out_csv=신용잔고_대주잔고_20230102_20251231.csv
[1/287] 005930 삼성전자 ...
  -> OK rows=731 date_min=2023-01-02 date_max=2025-12-30
[2/287] 373220 LG에너지솔루션 ...
  -> OK rows=731 date_min=2023-01-02 date_max=2025-12-30
[3/287] 207940 삼성바이오로직스 ...
  -> OK rows=731 date_min=2023-01-02 date_max=2025-12-30
[4/287] 000660 SK하이닉스 ...
  -> OK rows=731 date_min=2023-01-02 date_max=2025-12-30
[5/287] 051910 LG화학 ...
  -> OK rows=731 date_min=2023-01-02 date_max=2025-12-30
[6/287] 005935 삼성전자우 ...
  -> OK rows=731 date_min=2023-01-02 date_max=2025-12-30
[7/287] 006400 삼성SDI ...
  -> OK rows=731 date_min=2023-01-02 date_max=2025-12-30
[8/287] 005380 현대차 ...
  -> OK rows=731 date_min=2023-01-02 date_max=2025-12-30
[9/287] 035420 NAVER ...
  -> OK rows=731 date_min=2023-01-02 date_max=2025-12-30
[10/287] 000270 기아 ...
  -> OK rows=731 date_min=2023-01-02 date_max=2025-12-30
[11/287] 035720 카카오 ...
  -> OK ro